# Stage One: Classical ML baselines for CA prediction

**Goal.** Establish Random Forest and k-Nearest Neighbors baselines on the same prediction task later asked of LLMs:

> Given participant characteristics (demographics → employment → geography → transit), predict **group** and **interpersonal** communication-apprehension subscale scores on the PRCA 6–30 scale.

These baselines give a non-LLM reference for mean absolute error (MAE). Later LLM persona runs can be compared against the same tiered feature sets and the same ground-truth scores.

| Piece | Detail |
|---|---|
| Models | `RandomForestRegressor`, `KNeighborsRegressor` |
| Targets | `gt_group_ca`, `gt_interpersonal_ca` |
| Tiers | `demos`, `employment`, `geo`, `transit` (cumulative, matching LLM personas) |
| Validation | Leave-one-out when $N < 8$; otherwise 5-fold CV |
| Primary metric | MAE (aligned with LLM absolute error) |

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Support running from repo root or from notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from ca_personas.ml_baseline import (
    TIER_FEATURES,
    metrics_wide,
    prepare_modeling_frame,
    run_stage_one_baselines,
    save_baseline_artifacts,
)
from ca_personas.personas import TIERS

PROLIFIC = ROOT / "data" / "excerpts" / "prolific_excerpt.csv"
QUALTRICS = ROOT / "data" / "excerpts" / "qualtrics_excerpt.csv"
OUT_DIR = ROOT / "outputs" / "ml_baseline"

pd.set_option("display.max_columns", 50)
print("Project root:", ROOT)
print("Tiers:", TIERS)

## 2. Load joined participants and inspect modeling coverage

We reuse the project loader so ground-truth PRCA scoring matches the LLM evaluation path (Likert → 1–5, reverse-coded comfort items, subscale sums 6–30).

In [ ]:
participants, predictions, metrics = run_stage_one_baselines(
    PROLIFIC,
    QUALTRICS,
    tiers=TIERS,
    join_how="inner",
    n_neighbors=3,
    random_state=42,
)

print(f"Joined participants with complete CA targets: {len(participants)}")
participants[
    [
        "participant_id",
        "Age",
        "Sex",
        "Ethnicity simplified",
        "Country of residence",
        "Employment status",
        "LocationLatitude",
        "LocationLongitude",
        "gt_group_ca",
        "gt_interpersonal_ca",
    ]
]

In [ ]:
coverage = []
for tier in TIERS:
    frame = prepare_modeling_frame(participants, tier=tier)
    coverage.append(
        {
            "tier": tier,
            "n_modelable": len(frame),
            "n_features": len(TIER_FEATURES[tier]),
            "features": ", ".join(TIER_FEATURES[tier]),
        }
    )
pd.DataFrame(coverage)

## 3. Train / evaluate Random Forest and KNN

For each **tier × model × target**:

1. Build a sklearn `Pipeline` with median/mode imputation, one-hot encoding for categoricals, and scaling for numerics (important for KNN).
2. Generate out-of-fold predictions via cross-validation (LOOCV on the excerpt).
3. Clip predictions to the legal PRCA range $[6, 30]$.
4. Score MAE / RMSE / R².

The cell above already ran this end-to-end through `run_stage_one_baselines`.

In [ ]:
metrics.sort_values(["model", "tier", "target"])

In [ ]:
wide = metrics_wide(metrics)
wide

## 4. Visualize baseline MAE by tier

Lower MAE is better. These bars are the stage-one reference the LLM comparison will use.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
sides = [("mae_group", "Group CA"), ("mae_interpersonal", "Interpersonal CA")]
colors = {"random_forest": "#C5050C", "knn": "#333333"}

for ax, (col, title) in zip(axes, sides):
    for model, frame in wide.groupby("model"):
        ax.plot(frame["tier"], frame[col], marker="o", label=model, color=colors.get(model))
    ax.set_title(title)
    ax.set_xlabel("Persona / feature tier")
    ax.set_ylabel("MAE")
    ax.set_ylim(bottom=0)

axes[0].legend(frameon=False)
fig.suptitle("Stage-one ML baseline error by information tier", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, side, title in [
    (axes[0], "group", "Group CA"),
    (axes[1], "interpersonal", "Interpersonal CA"),
]:
    sub = predictions[predictions["side"] == side]
    for model, frame in sub.groupby("model"):
        ax.scatter(frame["y_true"], frame["y_pred"], alpha=0.75, label=model, s=50)
    ax.plot([6, 30], [6, 30], "--", color="gray", linewidth=1)
    ax.set_xlim(6, 30)
    ax.set_ylim(6, 30)
    ax.set_xlabel("Ground truth")
    ax.set_ylabel("CV prediction")
    ax.set_title(title)

axes[0].legend(frameon=False)
fig.suptitle("Predicted vs observed CA (all tiers pooled)", y=1.02)
fig.tight_layout()
plt.show()

## 5. Per-participant predictions

Long-form predictions are saved for later joins against LLM outputs (`participant_id`, `tier`, side).

In [ ]:
predictions.sort_values(["participant_id", "tier", "model", "side"]).head(24)

## 6. Persist artifacts

Writes:

- `outputs/ml_baseline/ml_baseline_predictions.csv`
- `outputs/ml_baseline/ml_baseline_metrics.csv`
- `outputs/ml_baseline/ml_baseline_metrics_wide.csv`

In [ ]:
paths = save_baseline_artifacts(predictions, metrics, OUT_DIR)
for name, path in paths.items():
    print(f"{name:14s} -> {path}")

## 7. How to read these baselines against the LLM

1. Run this notebook (or `run_stage_one_baselines`) whenever the participant extract updates.
2. Run the LLM persona pipeline for the same tiers (`ca-personas --provider ollama|openrouter`).
3. Compare `mae_group` / `mae_interpersonal` from `ml_baseline_metrics_wide.csv` to MAE in `outputs/evaluation/summary_by_tier.csv`.
4. If an LLM’s MAE is no better than RF/KNN on the same tier, the persona prompt is not beating a simple tabular learner on available covariates.

**Excerpt caveat.** With only a handful of inner-joined rows, LOOCV estimates are noisy. Re-run on the full cohort before drawing research conclusions; the API below does not change.